# 📱 Galaxy S25 FE - Instagram Reels Glitch Fixer
**Instructions (Works on Mobile & PC Browsers):**
1. Download the latest stock **Instagram `.apkm` (`arm64-v8a`)** from [APKMirror](https://www.apkmirror.com/apk/instagram/instagram-instagram/).
2. Tap the **Play button (▶️)** below, then tap **Choose Files** to upload your `.apkm` file.
3. *(For Future Updates Only)*: If you are updating Instagram and saved your `my-instagram.jks` file from an earlier run, select both your `.apkm` AND `my-instagram.jks` when uploading so your new APK can install directly over your current app without uninstalling!

In [ ]:
# @title ▶️ Tap the Play Button Here to Upload & Patch Instagram
import os, sys, glob, struct, zlib, hashlib, zipfile, subprocess, urllib.request
from google.colab import files

print('📂 Step 1: Tap "Choose Files" below and select your stock Instagram .apkm file')
print('   (If updating, also select your saved my-instagram.jks file):')
uploaded = files.upload()

APKEDITOR_URL = 'https://github.com/REAndroid/APKEditor/releases/download/V1.4.1/APKEditor-1.4.1.jar'
UBERSIGNER_URL = 'https://github.com/patrickfav/uber-apk-signer/releases/download/v1.3.0/uber-apk-signer-1.3.0.jar'
APKEDITOR_JAR = 'APKEditor.jar'
UBERSIGNER_JAR = 'uber-apk-signer.jar'
KEYSTORE_FILE = 'my-instagram.jks'

apkm_files = [f for f in uploaded.keys() if f.endswith(('.apkm', '.apk'))]
if not apkm_files:
    raise SystemExit('❌ No .apkm or .apk file was uploaded! Please run the cell again.')

input_file = apkm_files[0]
print(f'\n[*] Processing: {input_file}')

if not os.path.exists(APKEDITOR_JAR):
    urllib.request.urlretrieve(APKEDITOR_URL, APKEDITOR_JAR)
if not os.path.exists(UBERSIGNER_JAR):
    urllib.request.urlretrieve(UBERSIGNER_URL, UBERSIGNER_JAR)

if not os.path.exists(KEYSTORE_FILE):
    print('[*] Generating your unique personal signing key (my-instagram.jks)...')
    subprocess.run([
        'keytool', '-genkeypair', '-v', '-keystore', KEYSTORE_FILE,
        '-alias', 'igkey', '-keyalg', 'RSA', '-keysize', '2048', '-validity', '10000',
        '-storepass', 'instagram123', '-keypass', 'instagram123',
        '-dname', 'CN=User, OU=Mobile, O=Personal, L=City, S=State, C=US'
    ], check=True)
else:
    print('[*] Using your uploaded my-instagram.jks signing key!')

working_apk = 'merged_temp.apk'
if input_file.endswith('.apkm'):
    print('[*] Merging .apkm bundle into standalone APK...')
    subprocess.run(['java', '-jar', APKEDITOR_JAR, 'm', '-i', input_file, '-o', working_apk], check=True)
else:
    working_apk = input_file

def patch_dex(data: bytearray, name: str) -> bytes:
    modified = False
    for old, new in [(b'video/av01', b'video/av00'), (b'c2.exynos.', b'c2.android')]:
        if old in data:
            cnt = data.count(old)
            data = bytearray(bytes(data).replace(old, new))
            print(f'    [{name}] Patched {cnt}x: {old.decode()} -> {new.decode()}')
            modified = True
    if modified:
        data[12:32] = hashlib.sha1(data[32:]).digest()
        checksum = zlib.adler32(data[12:]) & 0xFFFFFFFF
        data[8:12] = struct.pack('<I', checksum)
    return bytes(data)

unaligned_apk = 'patched_unaligned.apk'
print('[*] Applying 10-byte Exynos AV1 decoder fix to DEX files...')
with zipfile.ZipFile(working_apk, 'r') as zin, zipfile.ZipFile(unaligned_apk, 'w') as zout:
    for item in zin.infolist():
        raw = zin.read(item.filename)
        if item.filename.startswith('classes') and item.filename.endswith('.dex'):
            raw = patch_dex(bytearray(raw), item.filename)
        zout.writestr(item, raw, compress_type=item.compress_type)

print('[*] Zipaligning & signing APK...')
subprocess.run([
    'java', '-jar', UBERSIGNER_JAR, '-a', unaligned_apk,
    '--ks', KEYSTORE_FILE, '--ksAlias', 'igkey',
    '--ksPass', 'instagram123', '--ksKeyPass', 'instagram123',
    '--allowResign', '--out', '.'
], check=True)

final_apk = 'instagram-s25fe-fixed.apk'
signed_out = 'patched_unaligned-aligned-signed.apk'
if os.path.exists(signed_out):
    os.replace(signed_out, final_apk)

print('\n✅ SUCCESS! Triggering browser download for your patched APK and personal key...')
files.download(final_apk)
files.download(KEYSTORE_FILE)